# Initial BIS data exploration

A descriptive review of the five project datasets: missing values, duplicate records, coverage, simple ranges, and charts. Monthly sample: **January 1994–August 2026**. USD credit stays quarterly from **2000Q1**, with expected completed quarters through **2026Q2**.

The brief does not list the 39 target currencies. This notebook therefore reviews the full downloaded universe and shows currency/economy overlaps. Illustrative chart currencies do not define the final sample. No filling, interpolation, peg exclusions, inferential statistics, or models are applied.

Source: [official BIS bulk exports](https://data.bis.org/bulkdownload). Run `python scripts/download_data.py` first, and use the environment in which the package is installed.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Markdown, Image
import pandas as pd

# Works from the repository root or notebooks/ without requiring an install.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "src" / "fx_forecasting").is_dir():
    raise RuntimeError("Open this notebook from the repository root or notebooks directory")
sys.path.insert(0, str(ROOT / "src"))
from fx_forecasting.data.eda import run_eda, area_values, UNITS

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
result = run_eda()
datasets, samples, tables = result["datasets"], result["samples"], result["tables"]
print("Generated report:", result["report"])
print("Raw exports are unchanged; descriptive matrices are in data/processed/.")

## 1. Sources, dimensions, and series definitions

Use native monthly observations and end-of-month FX. Broad real and nominal effective rates are separate matrices. CPI uses index levels rather than the alternative inflation-change series. GLI uses **total USD credit to non-banks**, including the outside-US aggregate and the 12 country series, in millions of USD.

The BIS nominal US effective-rate index is used as the dollar index. Quarterly credit is neither repeated into months nor lagged in this EDA.

In [ ]:
dimensions = pd.DataFrame([
    {"dataset": key, "frequency": data.values.index.freqstr,
     "periods_in_sample": len(samples[key]), "series": len(data.metadata),
     "units": UNITS[key]}
    for key, data in datasets.items()
])
display(dimensions)
for key in datasets:
    display(Markdown(f"**{key}: first three series and first three sample periods**"))
    display(samples[key].iloc[:3, :3])
display(datasets["usd_credit"].metadata[["BORROWERS_CTY", "Borrowers' country", "UNIT_MULT", "TITLE"]])

## 2. Duplicate and parsing checks

Each raw CSV row describes a series. Check all raw frequencies for exact repeated rows and repeated series identifiers. Series IDs and time headers must be unique before creating the series-period matrix. The same currency code in several countries (e.g. EUR) is a separate mapping issue, not an observation duplicate.

Blank fields and BIS `NaN` markers are missing values. Other nonnumeric entries are reported separately.

In [ ]:
display(tables["raw_audit"][["dataset", "raw_series_rows", "raw_frequencies", "raw_exact_duplicate_rows", "raw_duplicate_series_rows", "raw_blank_series_ids", "explicit_nan_tokens", "invalid_numeric_tokens"]])
display(tables["nonpositive_values"])
print("Policy rates are allowed to be zero or negative.")

## 3. Missing data and date coverage

Missingness uses the **complete expected calendar**, including periods absent from the export. Separate leading gaps, gaps within a series' observed history, and trailing gaps. Missing credit before 2000 is outside its intended sample. An incomplete 2026Q3 is not counted as a missing completed quarter.

The broad effective-rate series are complete. Policy gaps include stopped national euro-area histories and internal gaps in Japan. CPI has one internally missing US month. Credit is complete through 2026Q1, with 2026Q2 absent from this snapshot.

In [ ]:
display(tables["overview"])
profile = tables["missing_by_series"]
display(profile.nlargest(20, "missing_pct")[["dataset", "economy", "series", "missing_pct", "leading_missing", "internal_missing", "trailing_missing", "source_first", "source_last"]])
display(profile.loc[profile.internal_missing.gt(0), ["dataset", "economy", "internal_missing", "longest_internal_gap"]].sort_values("internal_missing", ascending=False))

In [ ]:
gaps = tables["missing_gap_dates"]
# Exact dates for internal gaps in macro predictors.
display(gaps[(gaps.gap_type == "internal") & gaps.dataset.isin(["policy", "cpi"])])
# Latest quarterly observation for every selected credit series.
display(profile[profile.dataset.eq("usd_credit")][["economy", "first_in_sample", "last_in_sample", "internal_missing", "trailing_missing"]])

In [ ]:
for name in ["coverage", "missing", "fx_missing_map"]:
    display(Image(filename=str(ROOT / "reports" / "figures" / f"eda_{name}.png")))

## 4. Currency and economy overlaps

The current exports have 64 broad EER economies and 63 CPI economies, compared with 63 and 62 in the brief. Reference-area matching is shown below. It does not resolve shared currencies, historical euro backcasts, the final 39-currency roster, or substitutions for national policy rates.

In [ ]:
overlap = tables["currency_overlap"]
print("All FX reference areas:", len(overlap), "| Distinct currency codes:", overlap.currency.nunique())
with_reer = overlap[overlap.has_reer]
print("FX reference areas with broad REER:", len(with_reer), "| Distinct currency codes:", with_reer.currency.nunique())
display(with_reer[["area", "economy", "currency", "has_policy", "has_cpi", "has_usd_credit"]].sort_values("economy").reset_index(drop=True))
display(tables["shared_currency_codes"][["area", "economy", "currency"]].sort_values(["currency", "economy"]).reset_index(drop=True))

## 5. Simple ranges and illustrative time paths

Counts, minimum, median, and maximum are calculated **within each series**. Units differ, so pooled raw means across currencies are not useful. FX paths use January 2010 = 100 and a logarithmic axis. Positive monthly FX changes indicate depreciation against the USD. Changes use adjacent calendar months with `fill_method=None`.

The illustrative policy-rate chart starts in 2000 so the very high early-1994 Brazilian rates do not flatten the other lines. Earlier observations remain in the data and range tables.

In [ ]:
examples = ["BR", "MX", "ZA", "TR", "CN", "JP", "GB", "US"]
display(profile[profile.area.isin(examples)][["dataset", "economy", "observed", "min", "median", "max", "units"]].sort_values(["dataset", "economy"]).reset_index(drop=True))
for name in ["fx_paths", "fx_changes", "macro", "credit"]:
    display(Image(filename=str(ROOT / "reports" / "figures" / f"eda_{name}.png")))

## 6. Large FX changes to review

These are descriptive review flags, not statistically identified outliers or confirmed errors. The largest moves in the full export can be far outside the intended 39-currency universe. Redenominations, regime changes, and historical source breaks should be checked before model preparation. No values are removed.

In [ ]:
display(tables["largest_fx_changes"][["period", "economy", "currency", "monthly_change_pct"]])

## Findings and next decisions

- No duplicate complete source rows or series identifiers; no unexpected nonnumeric observation tokens.
- Broad REER and NEER have no missing sample observations.
- Policy-rate missingness is largest and mixes late starts, stopped national histories, and internal gaps. The gap-date table provides exact periods to review.
- CPI gaps mainly reflect short histories and recent unavailable observations. The single internal US gap is left missing.
- All 13 credit series are complete through 2026Q1. Each lacks 2026Q2 in the downloaded snapshot.
- The full FX export contains 192 reference areas; it is not the final 39-currency sample.

Before modeling, confirm the currency roster, currency/economy mapping, peg exclusions, and emerging/advanced groups. Decide explicitly how to treat stopped euro-area policy histories and other missing predictors. Apply the planned two-quarter credit lag during feature construction, with a documented monthly availability rule. These are revised historical exports; reference dates alone do not establish publication-time availability.

The detailed written review is in `reports/eda/README.md`, quality tables in `reports/eda/`, selected matrices in `data/processed/`, and source URLs/hashes in `data/raw/download_manifest.json`.